# NZ 05 · Materialization and sweeps

Execute in numbered order. Existing DONE units are skipped with a reason.
Closed results are read-only; figures and hash manifests are saved under `results/_views/`.


In [ ]:
import os
from pathlib import Path
from IPython.display import display
import matplotlib.pyplot as plt
from sglib.core.infra.paths import find_repo_root
from sglib.dataoverview.handoff import load_bundle as load_dataoverview
from sglib.generator import stage
from sglib.generator.overview import tables, figures

REPO = find_repo_root(Path.cwd())
COUNTRY = "nz"
PROFILE = os.environ.get("SG_PROFILE", "formal")
RESULTS_ROOT = os.environ.get("SG_RESULTS_ROOT")
OPTIONS = dict(profile=PROFILE, results_root=RESULTS_ROOT, input_loader=load_dataoverview)
ctx = stage.country_context(REPO, COUNTRY, **OPTIONS)
print(f"backend={ctx.backend} results_root={ctx.results} read_only={ctx.read_only}")


In [ ]:
stage.run_step(REPO, COUNTRY, ['materialize', 'finalize', 'sweeps'], **OPTIONS)


In [ ]:
display(tables.candidates(ctx))
display(tables.inventory(ctx, ["sweeps"]))
fig, path = figures.candidate_coverage(ctx)
display(fig)
plt.close(fig)
print(path)


## Demand fields against land use

`weight_fields` is a 2x2 map grid for one analysis region: the dominant OSM land-use class per cell,
then three demand fields shown as density relative to the area-uniform field on a log2 scale (red above
uniform, blue below, grey = zero-support cells), stations overlaid. The first grid compares the static and
learned bases (GPM, MLP, GNN), the second the GNN prior/post/fusion NP variants. `concentration_curves`
(Lorenz) and `concentration_shares` (demand held by the densest 10 % of cells) quantify how strongly each
weighter concentrates demand. Set `REGION` to another configured region id to switch; learned families use
the first configured seed unless `seed=` is given.


In [ ]:
REGION = None  # None = first configured analysis region; set a region id to switch
display(tables.field_concentration(ctx, region=REGION))
fig, path = figures.weight_fields(ctx, region=REGION)
display(fig)
plt.close(fig)
print(path)


In [ ]:
region_name = figures._region_context(ctx, REGION)[1]
fig, path = figures.weight_fields(ctx, region=REGION, labels=("GNNpriorNP", "GNNpostNP", "GNNfusionNP"),
                                  name=f"05_weight_fields_{region_name}_gnn_variants.png")
display(fig)
plt.close(fig)
print(path)


In [ ]:
fig, path = figures.concentration_curves(ctx, region=REGION)
display(fig)
plt.close(fig)
print(path)
fig, path = figures.concentration_shares(ctx, region=REGION)
display(fig)
plt.close(fig)
print(path)
